# Rozsa et al. (2023) on DANDI:000341, part 2: RAF induction recordings

Paper: Rozsa, Toth, Olah, Baka, Lakovics, Barzo and Tamas (2023), Sci. Adv. 9, eade4511,
doi:10.1126/sciadv.ade4511. Claims file: `claims/rozsa-2023.yaml`, written from the version of
record (Europe PMC full text) and its supplement.

Dataset: DANDI:000341, version 0.241220.1312 (doi:10.48324/dandi.000341/0.241220.1312), the only
published version; the paper cites the dandiset without a version.

This notebook covers the folders `Human_persistent_RAF` (313 files) and `Rat_persistent_RAF`
(132 files), the RAF induction recordings. Each file is taken as one cell (checked for repeats
below).

| Claim | Here |
|---|---|
| fig1ab-raf-patterns | tested |
| fig1c-ra-lower-threshold-slope | tested |
| fig1d-human-raf-proportion | tested |
| fig1f-human-raf-ngfc | blocked, metadata_insufficient (no cell class in the files) |
| figs1b-rat-raf-proportion | tested |
| figs1d-rat-raf-values | tested |
| fig1g-rat-raf-ngfc | blocked, metadata_insufficient |
| fig2e-sag-species | tested |
| fig2e-sag-raf-smaller | tested |
| fig2il-kcl-induces-ra-aps | blocked, data_missing (no KCl application in any file's metadata) |

In [ ]:
import zlib, time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats
import raf

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
PAPER = "rozsa-2023"


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


T0 = time.time()
assets = raf.list_assets()
groups = collections.defaultdict(list)
for a in assets:
    groups[a["path"].split("/")[0]].append(a)
res = {}
for g in ("Human_persistent_RAF", "Rat_persistent_RAF"):
    t = time.time()
    res[g] = raf.process_group(groups[g], threads=2)
    print(f"{g}: {len(res[g])} files, {sum(a['size'] for a in groups[g]) / 1e9:.0f} GB in "
          f"{(time.time() - t) / 60:.1f} min; peak memory "
          f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Per-cell reduction

For each file: the induction sweeps (`raf.induction_sweeps`: shorter than 1 s, one positive step
of 50 to 300 ms), AIS-APs (APs during positive current) and RA-APs by reading (c) of the claims
file's header (`raf.ra_aps_induction`: APs without positive current after the first induction
sweep, rebound APs within 0.3 s of a hyperpolarizing step left out). `cont` marks the RA-APs of
the continuous recording (sweeps without any step), which is what the Methods count ("number of
RA-APs during continuous recording"). Reading (b) labels every AP of a species by a two-component
mixture on threshold and slope fitted to all its APs pooled. Sag from the sweeps with one
hyperpolarizing step, taken before the first induction sweep where there are any.

In [ ]:
cells = []
for g, sp in (("Human_persistent_RAF", "human"), ("Rat_persistent_RAF", "rat")):
    for p, (m, aps, sw, _) in res[g].items():
        aps = aps[aps.main] if len(aps) else aps
        ais, ra, t0, runs = raf.ra_aps_induction(aps, sw)
        stepless = set(sw[sw.nsteps == 0].series)
        ra = ra.assign(cont=ra.series.isin(stepless)) if len(ra) else ra.assign(cont=[])
        sag = sw.dropna(subset=["sag_a"]) if "sag_a" in sw else sw.iloc[:0]
        if len(sag) and np.isfinite(t0) and (sag.start < t0).any():
            sag = sag[sag.start < t0]
        v = sw[sw.chan.str.startswith("Vm")].sort_values("start")
        cells.append(dict(species=sp, file=p.split("/")[-1], subject=m["subject_subject_id"], meta=m,
                          aps=aps, ais=ais, ra=ra, t0=t0, runs=runs, sag=sag,
                          n_ind=len(raf.induction_sweeps(sw)), n_sweeps=len(v),
                          fingerprint=(len(v), tuple(v.vmed.round(3).head(20))),
                          rates=sorted(set(v.rate.round())), n_stim_pA=int(sw.get("stim_in_pA", pd.Series([False])).sum()),
                          n_stim_bad=int(sw.get("stim_bad", pd.Series([False])).sum())))
print(len(cells), "cells")

## Data check

In [ ]:
C = pd.DataFrame([{k: v for k, v in c.items() if k in ("species", "file", "subject", "t0", "n_ind", "n_sweeps",
                                                       "n_stim_pA", "n_stim_bad")}
                  | {"n_ais": len(c["ais"]), "n_ra": len(c["ra"]), "n_sag": len(c["sag"])} for c in cells])
for sp, d in C.groupby("species"):
    print(f"{sp}: files {len(d)}, subjects {d.subject.nunique()}, files with induction sweeps "
          f"{int((d.n_ind > 0).sum())}, with sag sweeps {int((d.n_sag > 0).sum())}; sweeps per file median "
          f"{d.n_sweeps.median():.0f}; stimulus stored in pA in {int((d.n_stim_pA > 0).sum())} files, "
          f"unusable in {int((d.n_stim_bad > 0).sum())}")
rates = collections.Counter(r for c in cells for r in c["rates"])
print("sampling rates present (files):", dict(rates))
allaps = pd.concat([c["aps"].assign(species=c["species"]) for c in cells if len(c["aps"])])
print(f"APs (peak > 0 mV): {len(allaps)}; threshold range {allaps.thr.min():.1f} to {allaps.thr.max():.1f} mV, "
      f"peak {allaps.peak.min():.1f} to {allaps.peak.max():.1f} mV, slope {allaps.slope.min():.1f} to {allaps.slope.max():.1f} mV/ms")
fp = collections.defaultdict(list)
for c in cells:
    fp[c["fingerprint"]].append(c["file"])
print("files with identical sweep fingerprints (same sweep count and median Vm of 20 sweeps):",
      [v for v in fp.values() if len(v) > 1] or "none")
words = collections.Counter()
for c in cells:
    for k, v in c["meta"]["pharmacology"].items():
        if v != "NaN" and "Name" in k:
            words[v] += 1
print("drugs named in general/pharmacology:", dict(words))
kcl = [c["file"] for c in cells if any(("kcl" in str(v).lower() or "potassium" in str(v).lower() or "k+" in str(v).lower())
                                       for v in list(c["meta"]["pharmacology"].values()) + [c["meta"].get("session_description"), c["meta"].get("experiment_description")])]
print("files whose metadata mention KCl or potassium:", kcl or "none")
ctype = [c["file"] for c in cells if any(k.lower().startswith(("celltype", "anatomy")) for k in c["meta"]["pharmacology"])]
print("files with a cell class field:", len(ctype))
print("session descriptions:", dict(collections.Counter(c["meta"]["session_description"] for c in cells)))

Reading (b): one mixture per species on all its APs.

In [ ]:
for sp in ("human", "rat"):
    idx = allaps.species == sp
    lab = raf.gmm_classes(allaps[idx], seed("fig1ab-raf-patterns") + (0 if sp == "human" else 1))
    allaps.loc[idx, "gmm_ra"] = lab
    print(sp, "mixture RA component:", int(lab.sum()), "of", int(idx.sum()), "APs; medians",
          allaps[idx].groupby(lab)[["thr", "slope"]].median().round(2).to_dict())
for c in cells:
    if len(c["ra"]):
        key = c["ra"].set_index(["series", "idx"]).index
        g = allaps[allaps.species == c["species"]]
        g = g[g.series.isin(c["ra"].series.unique())]
        lab = dict(zip(zip(g.series, g.idx), g.gmm_ra)) if len(g) else {}
        c["ra"] = c["ra"].assign(gmm=[bool(lab.get(k, False)) for k in key])
    else:
        c["ra"] = c["ra"].assign(gmm=[])


def cell_epochs(c, reading, which):
    """RAF epochs of a cell. which: 'cont' (continuous recording, reading (c)), 'all' (with the
    interstimulus intervals, reading (c)), 'gmm' (continuous recording, mixture-labelled RA-APs,
    reading (b)). reading: 'i' one epoch per cell, 'ii' one per induction run."""
    ra = c["ra"]
    if not len(ra):
        return []
    if which == "cont":
        ra = ra[ra.cont]
    elif which == "gmm":
        ra = ra[ra.cont & ra.gmm]
    return raf.epochs(ra.t_abs, None if reading == "i" else c["runs"])


def summarize(sp, reading, which):
    E = pd.DataFrame([e | {"file": c["file"]} for c in cells if c["species"] == sp
                      for e in cell_epochs(c, reading, which)])
    if not len(E):
        return E, {}
    E["persistent"] = E.freq > 5
    P, S = E[E.persistent], E[~E.persistent]
    out = dict(n_epochs=len(E), n_persistent=len(P), n_sporadic=len(S),
               pers_n_median=P.n.median(), pers_freq_median=P.freq.median(), pers_dur_median=P.duration.median(),
               pers_dur_mean=P.duration.mean(), pers_dur_min=P.duration.min(), pers_dur_max=P.duration.max(),
               spor_n_median=S.n.median(), spor_freq_median=S.freq.median(),
               all_dur_median=E.duration.median(), all_dur_mean=E.duration.mean(),
               all_dur_min=E.duration.min(), all_dur_max=E.duration.max(),
               pers_n_sd=P.n.std(), pers_freq_sd=P.freq.std(), spor_n_sd=S.n.std(), spor_freq_sd=S.freq.std())
    return E, out


def judge(out, exp, keys):
    hits = []
    for k, (v, tol) in zip(keys, exp):
        x = out.get(k, np.nan)
        hits.append(bool(np.isfinite(x) and abs(x - v) <= tol))
    return hits

## fig1ab-raf-patterns

> Human layer 1 interneurons show two forms of retroaxonal firing (RAF) after induction,
> persistent RAF with barrages of RA-APs at high frequency (median 88 RA-APs, 13.9 Hz; duration
> median 17.1 s, mean 68.4 s, from 0.18 to 2809 s) and sporadic RAF with a few RA-APs at low
> frequency (median 8 RA-APs, 0.31 Hz).

Readings: epochs (i) one per cell, (ii) one per induction run; RA-APs of the continuous
recording by reading (c), with the interstimulus intervals included as a variant, and the
mixture-labelled ones by reading (b). The durations are judged under persistent RAF and shown
over all epochs.

In [ ]:
EXP1 = [(88, 8.8), (13.9, 1.39), (17.1, 1.71), (68.4, 6.84), (0.18, 0.018), (2809, 280.9), (8, 0.8), (0.31, 0.031)]
K1 = ["pers_n_median", "pers_freq_median", "pers_dur_median", "pers_dur_mean", "pers_dur_min", "pers_dur_max",
      "spor_n_median", "spor_freq_median"]
rows = []
for which in ("cont", "all", "gmm"):
    for reading in ("i", "ii"):
        E, out = summarize("human", reading, which)
        h = judge(out, EXP1, K1)
        rows.append(dict(RA=which, epochs=reading, hits=sum(h), **{k: out.get(k, np.nan) for k in
                                                                   ["n_epochs", "n_persistent", "n_sporadic"] + K1},
                         pers_n_sd=out.get("pers_n_sd"), spor_freq_sd=out.get("spor_freq_sd"),
                         all_dur_median=out.get("all_dur_median"), all_dur_mean=out.get("all_dur_mean"),
                         all_dur_min=out.get("all_dur_min"), all_dur_max=out.get("all_dur_max")))
T1 = pd.DataFrame(rows)
print(T1.round(3).T.to_string())
print("expected:", dict(zip(K1, [e[0] for e in EXP1])))

**Result: not reproduced.** Persistent and sporadic epochs both occur, but at most two of the eight values are within tolerance under any reading (persistent mean duration 74.8 s and sporadic frequency 0.316 Hz, with the interstimulus intervals and one epoch per induction run); the first reading has none.

## fig1c-ra-lower-threshold-slope

> RA-APs have a lower somatic threshold potential and a lower pre-AP depolarization slope than
> AIS-APs, during persistent and during sporadic RAF.

Reading (c): AIS-APs during the current pulses, RA-APs after induction (interstimulus intervals
and continuous recording), within the cells whose one epoch (reading (i), RA-APs of the
continuous recording) is persistent or sporadic. APs pooled; the paper's rule (KS normality,
then t test or Mann-Whitney U). Per-cell medians as a variant.

In [ ]:
for sp in ("human", "rat"):
    for pattern in ("persistent", "sporadic"):
        A, B, PA, PB = [], [], [], []
        for c in cells:
            if c["species"] != sp:
                continue
            ep = cell_epochs(c, "i", "cont")
            if not ep:
                continue
            pers = ep[0]["freq"] > 5
            if pers != (pattern == "persistent"):
                continue
            A.append(c["ais"]); B.append(c["ra"])
            PA.append(c["ais"][["thr", "slope"]].median()); PB.append(c["ra"][["thr", "slope"]].median())
        if not A:
            print(sp, pattern, "no cells"); continue
        A, B = pd.concat(A), pd.concat(B)
        PA, PB = pd.DataFrame(PA), pd.DataFrame(PB)
        for q in ("thr", "slope"):
            t, p = raf.paper_test(B[q], A[q])
            tc = stats.mannwhitneyu(PB[q].dropna(), PA[q].dropna()).pvalue if len(PB[q].dropna()) and len(PA[q].dropna()) else np.nan
            print(f"{sp:5s} {pattern:10s} {q:5s}: cells {len(PA)}; AIS-AP median {A[q].median():7.2f} (n={len(A)}), "
                  f"RA-AP median {B[q].median():7.2f} (n={len(B)}); {t} P = {p:.2g}; per-cell medians "
                  f"{PA[q].median():.2f} vs {PB[q].median():.2f}, Mann-Whitney P = {tc:.2g}")

**Result: reproduced.** In human persistent cells RA-APs have a lower threshold (-59.8 against -44.2 mV) and slope (0.85 against 1.89 mV/ms), and in sporadic cells (-61.2 against -44.5 mV, 0.82 against 2.25 mV/ms), each with Mann-Whitney P below 1e-300; per-cell medians agree (P <= 0.0022).

## fig1d-human-raf-proportion and figs1b-rat-raf-proportion

> About one-fourth of tested human layer 1 interneurons (87 of 385) show RAF. About a third of
> tested rat layer 1 interneurons (102 of 259) show RAF.

A cell shows RAF when at least one RA-AP occurs after induction. Tested cells are the files with
induction sweeps; all files are shown as the other denominator. RA-APs by reading (c) (with or
without the interstimulus intervals) and by reading (b) (reading (c) RA-APs the mixture also
labels RA). At least two RA-APs is shown as a variant against single spurious events.

In [ ]:
prop = []
for c in cells:
    ra = c["ra"]
    prop.append(dict(species=c["species"], tested=c["n_ind"] > 0,
                     c_all=len(ra), c_cont=int(ra.cont.sum()) if len(ra) else 0,
                     b_all=int(ra.gmm.sum()) if len(ra) else 0))
prop = pd.DataFrame(prop)
for sp, exp in (("human", 0.226), ("rat", 0.394)):
    d = prop[prop.species == sp]
    t = d[d.tested]
    for col, name in (("c_all", "reading (c), ISIs and continuous"), ("c_cont", "reading (c), continuous only"),
                      ("b_all", "reading (b)")):
        for k in (1, 2):
            n = int((t[col] >= k).sum())
            print(f"{sp:5s} {name:34s} >= {k} RA-AP: {n} of {len(t)} tested = {n / max(len(t), 1):.3f}; "
                  f"of all {len(d)} files = {n / len(d):.3f}")
    print(f"   expected {exp} (tolerance 10%)")

**Result (fig1d-human-raf-proportion): not reproduced.** 302 of 307 human cells (0.984) have RA-APs after induction, 168 (0.547) on the continuous recording and 162 (0.528) under the mixture, against 0.226.

**Result (figs1b-rat-raf-proportion): not reproduced.** 115 of 118 rat cells (0.975) have RA-APs after induction, 99 (0.839) on the continuous recording and 97 (0.822) under the mixture, against 0.394.

## figs1d-rat-raf-values

> Rat interneurons show both sporadic RAF (frequency 1 Hz, 7 RA-APs) and tonic RAF (frequency
> 6.9 Hz, 100 RA-APs).

In [ ]:
EXP2 = [(1, 0.1), (7, 0.7), (6.9, 0.69), (100, 10)]
K2 = ["spor_freq_median", "spor_n_median", "pers_freq_median", "pers_n_median"]
rows = []
for which in ("cont", "all", "gmm"):
    for reading in ("i", "ii"):
        E, out = summarize("rat", reading, which)
        rows.append(dict(RA=which, epochs=reading, hits=sum(judge(out, EXP2, K2)),
                         **{k: out.get(k, np.nan) for k in ["n_epochs", "n_persistent", "n_sporadic"] + K2}))
print(pd.DataFrame(rows).round(3).T.to_string())
print("expected:", dict(zip(K2, [e[0] for e in EXP2])))

**Result: not reproduced.** At most one of the four rat values is within tolerance under any reading.

## fig2e-sag-species and fig2e-sag-raf-smaller

> Human layer 1 interneurons have a higher somatic sag fraction than rat ones (0.15 against
> 0.04). In both species, interneurons with RAF have smaller somatic sag fractions than those
> without (human 0.11 against 0.16; rat 0.02 against 0.06).

Per cell, the sag of one step sweep: the most hyperpolarizing step, or the step whose peak is
nearest -90 mV; formula (a) or (b) (`raf._sag`). RAF status by reading (c), RA-APs with the
interstimulus intervals, at least one RA-AP, over all files with sag sweeps.

In [ ]:
sg = []
for c, pr in zip(cells, prop.itertuples()):
    s = c["sag"]
    if not len(s):
        continue
    most = s[s.i_step == s.i_step.min()].iloc[0]
    near = s.iloc[int(np.argmin(np.abs(s.vpeak.to_numpy() + 90)))]
    sg.append(dict(species=c["species"], file=c["file"], raf=pr.c_all > 0, tested=pr.tested,
                   a_most=most.sag_a, b_most=most.sag_b, a_90=near.sag_a, b_90=near.sag_b,
                   step_most=most.i_step, peak_most=most.vpeak))
sg = pd.DataFrame(sg)
print(sg.groupby("species")[["a_most", "b_most", "a_90", "b_90", "step_most", "peak_most"]].describe().T.round(3))

In [ ]:
for col in ("a_most", "b_most", "a_90", "b_90"):
    h, r = sg[sg.species == "human"][col].dropna(), sg[sg.species == "rat"][col].dropna()
    p = stats.mannwhitneyu(h, r).pvalue
    print(f"{col:7s}: human n={len(h)} median {h.median():.3f} (SD {h.std():.3f}); rat n={len(r)} median "
          f"{r.median():.3f} (SD {r.std():.3f}); Mann-Whitney P = {p:.2g}")
print("expected: human 0.15 (n=358), rat 0.04 (n=254), tolerance 10%, P < 0.001")

**Result: partially reproduced.** Human sag is higher than rat (0.198 against 0.071, Mann-Whitney P = 1.5e-26, 312 and 129 cells), but neither median is within tolerance of 0.15 and 0.04.

In [ ]:
for col in ("a_most", "b_most", "a_90", "b_90"):
    for sp in ("human", "rat"):
        d = sg[sg.species == sp]
        y, n = d[d.raf][col].dropna(), d[~d.raf][col].dropna()
        p = stats.mannwhitneyu(y, n).pvalue if len(y) and len(n) else np.nan
        print(f"{col:7s} {sp:5s}: RAF n={len(y)} median {y.median():.3f}; no RAF n={len(n)} median {n.median():.3f}; "
              f"Mann-Whitney P = {p:.2g}")
print("expected: human 0.11 (n=83) vs 0.16 (n=275); rat 0.02 (n=97) vs 0.06 (n=157); P < 0.001")
print(f"notebook time {(time.time() - T0) / 60:.1f} min, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

**Result: not reproduced.** Human RAF and no-RAF cells do not differ (0.198 against 0.199, P = 0.8); in rat RAF cells are lower (0.062 against 0.125) with P = 0.0028, not P < 0.001.

## Summary

| Claim | Outcome |
|---|---|
| fig1ab-raf-patterns | not reproduced |
| fig1c-ra-lower-threshold-slope | reproduced |
| fig1d-human-raf-proportion | not reproduced |
| fig1f-human-raf-ngfc | blocked, metadata_insufficient |
| figs1b-rat-raf-proportion | not reproduced |
| figs1d-rat-raf-values | not reproduced |
| fig1g-rat-raf-ngfc | blocked, metadata_insufficient |
| fig2e-sag-species | partially reproduced |
| fig2e-sag-raf-smaller | not reproduced |
| fig2il-kcl-induces-ra-aps | blocked, data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Induction sweeps, AIS-APs and RA-APs are rebuilt from the stimulus and the traces (reading (c) of the claims file), as the files carry no labels; epochs are one per cell or one per induction run, as the claims file names. Sag is taken on the step sweeps before induction with a formula the paper does not give (two readings). The seed range of the mixture reading was not run.

What was found about the dataset. The folders hold 313 human and 132 rat files against 385 and 259 tested cells, and nearly every cell in them fires APs without current after induction, so they appear to be a selection the dandiset does not describe. No file names a cell class or a KCl application. Sampling rates range from 5 to 200 kHz, and the gap-free recordings after induction are often at 5 kHz.
